In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


# =========================
# Load Data
# =========================

train = pd.read_csv("train.csv")
test = pd.read_csv("public_test.csv")

test_ids = test["id"]


# =========================
# Feature Engineering
# =========================

def create_features(df):

    df = df.copy()

    # IO activity
    df["io_total"] = df["rchar"] + df["wchar"]

    # Memory activity
    df["mem_total"] = df["lread"] + df["lwrite"]

    # Ratios
    df["read_ratio"] = df["rchar"] / (df["wchar"] + 1)
    df["mem_ratio"] = df["lread"] / (df["lwrite"] + 1)

    # System calls
    df["sys_total"] = df["sread"] + df["swrite"]
    df["sys_ratio"] = df["sread"] / (df["swrite"] + 1)

    # Process behaviour
    df["proc_total"] = df["fork"] + df["exec"]

    # System pressure
    df["system_pressure"] = df["runqsz"] * df["scall"]

    # Memory pressure
    df["memory_pressure"] = df["freemem"] / (df["freeswap"] + 1)

    # Safe log transform
    for col in ["rchar","wchar","scall","lread","lwrite"]:
        if col in df.columns:
            df["log_"+col] = np.log1p(df[col].clip(lower=0))

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    return df


train = create_features(train)
test = create_features(test)


# =========================
# Prepare Data
# =========================

X = train.drop(["usr","id"],axis=1)
y = train["usr"]

X_test = test.drop(["id"],axis=1)


# =========================
# Encode categorical columns
# =========================

for col in X.select_dtypes(include="object").columns:

    le = LabelEncoder()

    X[col] = le.fit_transform(X[col])
    X_test[col] = le.transform(X_test[col])


# =========================
# Fill Missing Values
# =========================

num_cols = X.select_dtypes(include=[np.number]).columns

X[num_cols] = X[num_cols].fillna(X[num_cols].mean())
X_test[num_cols] = X_test[num_cols].fillna(X_test[num_cols].mean())


# =========================
# Cross Validation
# =========================

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros(len(X))
preds = np.zeros(len(X_test))


# =========================
# Training Loop
# =========================

for fold,(train_idx,val_idx) in enumerate(kf.split(X),1):

    print("Fold:",fold)

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    # XGBoost
    xgb = XGBRegressor(
        n_estimators=1200,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=42
    )

    # LightGBM
    lgb = LGBMRegressor(
        n_estimators=1200,
        learning_rate=0.03,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    # CatBoost
    cat = CatBoostRegressor(
        iterations=1200,
        learning_rate=0.03,
        depth=6,
        verbose=0,
        random_state=42
    )

    # Train
    xgb.fit(X_train,y_train)
    lgb.fit(X_train,y_train)
    cat.fit(X_train,y_train)

    # Validation prediction
    val_pred = (
        0.3*xgb.predict(X_val) +
        0.5*lgb.predict(X_val) +
        0.2*cat.predict(X_val)
    )

    val_pred = np.clip(val_pred,0,100)

    oof[val_idx] = val_pred

    # Test prediction
    test_pred = (
        0.3*xgb.predict(X_test) +
        0.5*lgb.predict(X_test) +
        0.2*cat.predict(X_test)
    )

    test_pred = np.clip(test_pred,0,100)

    preds += test_pred/5

    rmse = np.sqrt(mean_squared_error(y_val,val_pred))

    print("RMSE:",rmse)


# =========================
# Overall CV Score
# =========================

rmse = np.sqrt(mean_squared_error(y,oof))

print("Overall CV RMSE:",rmse)


# =========================
# Submission File
# =========================

submission = pd.DataFrame({

    "id":test_ids,
    "usr":preds
})

submission.to_csv("submission2.csv",index=False)

print("submission2.csv created successfully")

Fold: 1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001745 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7840
[LightGBM] [Info] Number of data points in the train set: 3931, number of used features: 35
[LightGBM] [Info] Start training from score 84.200967
RMSE: 2.1200404751888016
Fold: 2
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.020964 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7842
[LightGBM] [Info] Number of data points in the train set: 3931, number of used features: 35
[LightGBM] [Info] Start training from score 84.167133
RMSE: 1.6093345612044958
Fold: 3
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002651 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7842
[LightGBM] [Info] Number of data points in the